In [ ]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from textblob import TextBlob
from sklearn.decomposition import NMF
from scipy.sparse import save_npz

In [2]:
processed_data_path = Path("../data/processed/processed.csv")
df = pd.read_csv(processed_data_path)
df.head()

,review,rating
0,absolutely wonderful silky sexy comfortable,4
1,love dress sooo pretty happen find store im gl...,5
2,high hope dress really want work initially ord...,3
3,love love love jumpsuit fun flirty fabulous ev...,5
4,shirt flattering due adjustable front tie perf...,5


In [3]:
df.shape

(22641, 2)

In [4]:
df['rating'].value_counts().sort_index()

rating
1      821
2     1549
3     2823
4     4908
5    12540
Name: count, dtype: int64

In [5]:
def create_sentiment_labels(rating):
    if rating <= 2:
        return 'negative'
    elif rating == 3:
        return 'neutral'
    else:
        return 'positive'

df['sentiment'] = df['rating'].apply(create_sentiment_labels)
df['sentiment'].value_counts()

sentiment
positive    17448
neutral      2823
negative     2370
Name: count, dtype: int64

In [6]:
bow_vectorizer = CountVectorizer(max_features=500)
X_bow = bow_vectorizer.fit_transform(df['review'])
X_bow.shape

(22641, 500)

In [7]:
tfidf_vectorizer = TfidfVectorizer(max_features=500)
X_tfidf = tfidf_vectorizer.fit_transform(df['review'])
X_tfidf.shape

(22641, 500)

In [8]:
X_train_bow, X_test_bow, y_train, y_test = train_test_split(X_bow, df['sentiment'], test_size=0.2, random_state=42)
X_train_tfidf, X_test_tfidf, _, _ = train_test_split(X_tfidf, df['sentiment'], test_size=0.2, random_state=42)
print(f"Training set size: {X_train_bow.shape[0]}")
print(f"Test set size: {X_test_bow.shape[0]}")

Training set size: 18112
Test set size: 4529


In [9]:
dev_data_dir = Path("../data/development/")
dev_data_dir.mkdir(parents=True, exist_ok=True)

save_npz(dev_data_dir / "X_train_bow.npz", X_train_bow)
save_npz(dev_data_dir / "X_test_bow.npz", X_test_bow)
save_npz(dev_data_dir / "X_train_tfidf.npz", X_train_tfidf)
save_npz(dev_data_dir / "X_test_tfidf.npz", X_test_tfidf)

y_train.to_csv(dev_data_dir / "y_train.csv", index=False, header=['sentiment'])
y_test.to_csv(dev_data_dir / "y_test.csv", index=False, header=['sentiment'])

with open(dev_data_dir / "bow_vectorizer.pkl", 'wb') as f:
    pickle.dump(bow_vectorizer, f)
with open(dev_data_dir / "tfidf_vectorizer.pkl", 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

In [10]:
nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)
nb_bow_score = nb_bow.score(X_test_bow, y_test)
nb_bow_score

0.7990726429675425

In [11]:
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)
nb_tfidf_score = nb_tfidf.score(X_test_tfidf, y_test)
nb_tfidf_score

0.7822918966659307

In [12]:
lr_bow = LogisticRegression(max_iter=1000, random_state=42)
lr_bow.fit(X_train_bow, y_train)
lr_bow_score = lr_bow.score(X_test_bow, y_test)
lr_bow_score

0.8116582026937513

In [13]:
lr_tfidf = LogisticRegression(max_iter=1000, random_state=42)
lr_tfidf.fit(X_train_tfidf, y_train)
lr_tfidf_score = lr_tfidf.score(X_test_tfidf, y_test)
lr_tfidf_score

0.8136453963347318

In [14]:
comparison_data = {
    'Model': ['Naive Bayes', 'Naive Bayes', 'Logistic Regression', 'Logistic Regression'],
    'Vectorization': ['Bag of Words', 'TF-IDF', 'Bag of Words', 'TF-IDF'],
    'Accuracy': [nb_bow_score, nb_tfidf_score, lr_bow_score, lr_tfidf_score]
}
comparison_df = pd.DataFrame(comparison_data)
comparison_df.sort_values(by='Accuracy', ascending=False)

,Model,Vectorization,Accuracy
3,Logistic Regression,TF-IDF,0.813645
2,Logistic Regression,Bag of Words,0.811658
0,Naive Bayes,Bag of Words,0.799073
1,Naive Bayes,TF-IDF,0.782292


In [15]:
def textblob_sentiment(text):
    polarity = TextBlob(text).sentiment.polarity
    if polarity < -0.1:
        return 'negative'
    elif polarity > 0.1:
        return 'positive'
    else:
        return 'neutral'

df['textblob_sentiment'] = df['review'].apply(textblob_sentiment)
textblob_accuracy = (df['textblob_sentiment'] == df['sentiment']).sum() / len(df)
textblob_accuracy

np.float64(0.720418709420962)

In [16]:
intent_keywords = {
    'complaint': ['problem', 'issue', 'wrong', 'broken', 'defect', 'poor', 'bad', 'terrible', 'awful', 'horrible', 'disappointing'],
    'refund': ['refund', 'return', 'money', 'exchange', 'back'],
    'delivery': ['deliver', 'shipping', 'arrive', 'package', 'delayed', 'late', 'delivery', 'shipped'],
    'general': ['love', 'great', 'excellent', 'good', 'nice', 'beautiful', 'comfortable', 'perfect']
}

def classify_intent(review):
    text = review.lower()
    scores = {intent: 0 for intent in intent_keywords}
    for intent, keywords in intent_keywords.items():
        scores[intent] = sum(text.count(keyword) for keyword in keywords)
    return max(scores, key=scores.get)

df['intent'] = df['review'].apply(classify_intent)
df['intent'].value_counts()

intent
general      15680
complaint     4013
refund        2650
delivery       298
Name: count, dtype: int64

In [17]:
nmf_model = NMF(n_components=5, random_state=42, max_iter=500)
nmf_features = nmf_model.fit_transform(X_tfidf)
nmf_features.shape

(22641, 5)

In [18]:
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
for topic_idx, topic in enumerate(nmf_model.components_):
    top_indices = topic.argsort()[-10:][::-1]
    top_words = feature_names[top_indices]
    print(f"Topic {topic_idx}: {' '.join(top_words)}")

Topic 0: size small order run large fit im medium usually true
Topic 1: dress wear beautiful love perfect slip fit flatter comfortable make
Topic 2: love great wear color jean comfortable buy soft perfect fit
Topic 3: look like really fabric make would back think nice good
Topic 4: top love cute bottom wear pretty beautiful tank detail white


In [19]:
df['topic'] = nmf_features.argmax(axis=1)
df['topic'].value_counts().sort_index()

topic
0    2589
1    3368
2    6916
3    7121
4    2647
Name: count, dtype: int64

In [20]:
models_dir = Path("../models/")
models_dir.mkdir(parents=True, exist_ok=True)

with open(models_dir / "sentiment_model.pkl", 'wb') as f:
    pickle.dump(lr_tfidf, f)
with open(models_dir / "intent_keywords.pkl", 'wb') as f:
    pickle.dump(intent_keywords, f)
with open(models_dir / "nmf_model.pkl", 'wb') as f:
    pickle.dump(nmf_model, f)

In [21]:
with open(models_dir / "nb_bow.pkl", 'wb') as f:
    pickle.dump(nb_bow, f)

with open(models_dir / "nb_tfidf.pkl", 'wb') as f:
    pickle.dump(nb_tfidf, f)

with open(models_dir / "lr_bow.pkl", 'wb') as f:
    pickle.dump(lr_bow, f)

with open(models_dir / "lr_tfidf.pkl", 'wb') as f:
    pickle.dump(lr_tfidf, f)